# Build three agents

You'll build a Personalised Tutor Agent, a Flashcard Generator Agent and a Study Planner Agent. The tools and the fictional data are already wired up, so most of the work is deciding what to tell each agent to do.

Each one follows the same four steps:

1. **Inspect the tools** the agent has access to.
2. **Write CRAFT instructions** in the `_INSTRUCTIONS` cell.
3. **Run it** and give it a task.
4. **Read the trace** in the checkpoint questions, then go back and change something.

The three sections below are independent. You don't need to finish one before starting the next.

> All deadlines, calendars, progress records, course notes and actions in this notebook are workshop simulations. Each agent can support studying and planning, but must not produce assessed work for submission.

## Setup

Run these cells once. The root `.env` is used only by this local notebook/runtime. If no key is present, the next cell asks for one without writing it into the notebook.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
from getpass import getpass
from dotenv import load_dotenv
import os

load_dotenv('.env', override=True)
if not os.getenv('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass('Anthropic API key: ')

from workshopkit import *

---
## Agent 1: Personalised Tutor Agent

**Goal:** `Explain a leadership or organisational behaviour concept using only the supplied unit material.`

The agent should search the course documents before answering, name what it used, and say plainly when the material doesn't cover something instead of guessing.

In [ ]:
# Inspect the tutor's tool before writing the agent instructions.
for tool in TUTOR_TOOLS:
    print(f'\n{tool.name}: {tool.description}')
    print(tool.input_schema)

print('\nDirect retrieval check:')
print(search_course_notes.execute({
    'query': 'What is the difference between transformational and transactional leadership?',
    'unit_code': 'MGMT2002',
    'top_k': 2,
}))

In [ ]:
# YOUR TURN: change at least one line below (try the Approach line first), then run the cell.
TUTOR_INSTRUCTIONS = """
C: Context. You support a university student using fictional workshop course documents for MGMT2002.
R: Request. Explain the concept the student asks about, using only the supplied material.
A: Approach. Search the course documents before answering. Name the source you used. If the documents don't cover the question, say so rather than filling the gap yourself.
F: Format and constraints. Keep the explanation short and plain. Do not invent lecture content. Do not write the student's assessment for them.
T: Test. The explanation matches what the retrieved passage actually says, the source is named, and any gap in the material is named rather than papered over.
"""

TUTOR_TASK = "Can you explain situational leadership and how it's different from a transformational approach?"

tutor_result = run_tutor_agent(TUTOR_INSTRUCTIONS, TUTOR_TASK)
show_trace(tutor_result, 'AGENT 1: TUTOR TRACE')

### Trace checkpoint

Check the observable evidence:

- Did it search before answering, or answer straight away?
- Does the explanation match what the retrieved passage actually says?
- Did it name the source file it used?
- What would you change in CRAFT before changing any code?

---
## Agent 2: Flashcard Generator Agent

**Goal:** `Turn what I'm weakest on into a few practice flashcards.`

The agent should check the mastery record to see which topics need work, retrieve a supporting passage, then generate flashcards grounded in that passage.

In [ ]:
for tool in FLASHCARD_TOOLS:
    print(f'\n{tool.name}: {tool.description}')

print('\nMastery record check:')
print(get_mastery_record.execute({}))

In [ ]:
# YOUR TURN: change at least one line below (try the topic in the task), then run the cell.
FLASHCARD_INSTRUCTIONS = """
C: Context. You support a university student revising for MGMT2002 using fictional workshop mastery data and course documents.
R: Request. Turn a weak topic into a small set of practice flashcards.
A: Approach. Check the mastery record to find the weakest relevant topic, search the course documents for a supporting passage, then generate flashcards grounded in that passage. Do not generate cards for a topic you haven't retrieved material for.
F: Format and constraints. Return a short set of front and back flashcards, plus which topic and source they came from. These are for self-study only, not assessed answers.
T: Test. The chosen topic is genuinely one of the weaker ones, the cards are grounded in the retrieved passage, and the source is named.
"""

FLASHCARD_TASK = "I have 20 minutes. Make me a few flashcards on whatever I'm weakest on."

flashcard_result = run_flashcard_agent(FLASHCARD_INSTRUCTIONS, FLASHCARD_TASK)
show_trace(flashcard_result, 'AGENT 2: FLASHCARD TRACE')

### Trace checkpoint

- Did it check mastery before picking a topic, or guess?
- Is the topic it picked actually one of the weaker ones?
- Are the cards grounded in the retrieved passage, or generic?
- Would you trust these cards without checking them yourself?

---
## Agent 3: Study Planner Agent

**Goal:** `Plan what I should work on this week.`

The agent should check deadlines, progress and realistic capacity before proposing a plan. It must protect buffer time and ask before saving anything.

In [ ]:
# Inspect the five tool contracts before writing the agent instructions.
for tool in PLANNER_TOOLS:
    print(f'\n{tool.name}: {tool.description}')
    print(tool.input_schema)

In [ ]:
# YOUR TURN: change at least one line below (try the Format and constraints line), then run the cell.
PLANNER_INSTRUCTIONS = """
C: Context. You support a university student using fictional workshop deadline, calendar and progress data.
R: Request. Work out what the student should focus on this week and create a realistic plan.
A: Approach. Fetch missing planning facts with tools, prioritise urgency and remaining workload, and ask rather than guess when critical information is absent.
F: Format and constraints. Return priorities, a day-by-day plan, risks and what to start tonight. Protect rest, include buffer, do not write assessed content, and do not save anything without approval.
T: Test. Every deadline is addressed, scheduled work fits available hours, progress changes the priorities, buffer remains, and all simulated data is labelled.
"""

PLANNER_TASK = "Plan what I should work on this week."

planner_result = run_study_planner(PLANNER_INSTRUCTIONS, PLANNER_TASK)
show_trace(planner_result, 'AGENT 3: STUDY PLANNER TRACE')

### Trace checkpoint

Check the observable evidence:

- Which facts did the agent fetch instead of inventing?
- Did current progress change the priority order?
- Did the work fit the available blocks and leave buffer?
- Did it avoid `save_study_plan()` unless the student approved an action?
- What would you change in CRAFT before changing any code?

---

You've now built three agents from the same four-step pattern: inspect the tools, write CRAFT instructions, run it, read the trace. Same shape, three different jobs.

## Make it yours (stretch goal)

Choose a useful direction: Study Planner, Research Assistant, Revision Coach, Assignment Reviewer, Group Project Coordinator, Career or Application Assistant, or something else. Define the goal, necessary context, tools, retrieval, specialist roles and approval boundaries before adding code.

In [ ]:
MY_AGENT = {
    'goal': '...',
    'context_needed': ['...'],
    'tools_needed': ['...'],
    'retrieval_needed': False,
    'specialist_needed': False,
    'human_approval_before': ['...'],
    'success_test': ['...'],
}

MY_AGENT